In [1]:
import pandas as pd
import numpy as np

print("Début du chargement des fichiers d'Olist...")

# 1. CHARGEMENT DES FICHIERS DONT ON A BESOIN
orders = pd.read_csv('../data/olist_orders_dataset.csv')
items = pd.read_csv('../data/olist_order_items_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
translations = pd.read_csv('../data/product_category_name_translation.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv') # <-- AJOUT POUR LE PROF

print("Tous les fichiers ont été chargés avec succès !")

# 2. TRADUCTION DES CATÉGORIES EN ANGLAIS
products = products.merge(translations, on='product_category_name', how='left')
products['product_category'] = products['product_category_name_english'].fillna(products['product_category_name']).fillna('other')

# 3. TRAITEMENT DES REVIEWS (On gère les doublons s'il y a plusieurs avis par commande)
reviews_clean = reviews.groupby('order_id')['review_score'].mean().reset_index()

# 4. FUSION (MERGE) DES TABLES
df = orders.merge(items, on='order_id', how='inner')
df = df.merge(customers, on='customer_id', how='inner')
df = df.merge(products, on='product_id', how='inner')
df = df.merge(reviews_clean, on='order_id', how='left') # <-- AJOUT DES REVIEWS

# 5. NETTOYAGE DES DONNÉES
df = df[df['order_status'] == 'delivered']
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['total_price'] = df['price'] + df['freight_value']
df['quantity'] = 1

# Si une commande n'a pas de note, on met la note médiane (5) par sécurité
df['review_score'] = df['review_score'].fillna(5)

# Calcul de la recency globale pour la suite (par rapport à la date max du dataset)
max_date = df['order_purchase_timestamp'].max()
df['recency'] = (max_date - df['order_purchase_timestamp']).dt.days

# 6. SÉLECTION DES COLONNES ESSENTIELLES
columns_to_keep = [
    'customer_unique_id', 
    'customer_state',     
    'order_id',           
    'order_purchase_timestamp',
    'recency',            
    'product_id',         
    'product_category',   
    'quantity',           
    'price',              
    'total_price',        
    'freight_value',
    'review_score'        # <-- AJOUT POUR TON CLUSTERING
]

final_df = df[columns_to_keep]

print("\n--- NETTOYAGE TERMINÉ ---")
print(f"Nombre total de lignes exploitables : {final_df.shape[0]}")

# ==========================================
# 7. L'ÉTAPE MAGIQUE : LA SAUVEGARDE EN CSV
# ==========================================
final_df.to_csv('../data/olist_cleaned_raw.csv', index=False)
print("\n💾 Le fichier 'olist_cleaned_raw.csv' a été sauvegardé avec succès dans ton dossier data !")

Début du chargement des fichiers d'Olist...
Tous les fichiers ont été chargés avec succès !

--- NETTOYAGE TERMINÉ ---
Nombre total de lignes exploitables : 110197

💾 Le fichier 'olist_cleaned_raw.csv' a été sauvegardé avec succès dans ton dossier data !


In [2]:
import pandas as pd

# 1. Chargement de la base transactionnelle brute qu'on a créée ensemble
df_raw = pd.read_csv('../data/olist_cleaned_raw.csv')

# 2. Agrégation par client unique pour ajouter la colonne demandée par le prof
df_clus_with_review = df_raw.groupby('customer_unique_id').agg(
    recency=('recency', 'first'),
    frequency=('order_id', 'nunique'),
    monetary_value=('total_price', 'sum'),
    mean_freight=('freight_value', 'mean'),
    review_score_mean=('review_score', 'mean'), # <-- La nouvelle colonne ajustée !
    customer_state=('customer_state', 'first')   # On garde l'État pour l'encodage plus tard
).reset_index()

# 3. Sauvegarde de cette étape intermédiaire
df_clus_with_review.to_csv('../data/olist_clustering_with_review.csv', index=False)

print("--- ÉTAPE 1 TERMINÉE ---")
print(f"Le dataset de clustering contient maintenant la note moyenne des avis.")
print(f"Dimensions : {df_clus_with_review.shape[0]} clients et {df_clus_with_review.shape[1]} colonnes.")
print(df_clus_with_review[['customer_unique_id', 'monetary_value', 'review_score_mean']].head(3))

--- ÉTAPE 1 TERMINÉE ---
Le dataset de clustering contient maintenant la note moyenne des avis.
Dimensions : 93358 clients et 7 colonnes.
                 customer_unique_id  monetary_value  review_score_mean
0  0000366f3b9a7992bf8c76cfdf3221e2          141.90                5.0
1  0000b849f77a49e4a4ce2b2a4ca5be3f           27.19                4.0
2  0000f46a3911fa3c0805444483337064           86.22                3.0


In [3]:
df_clus_with_review.head()

,customer_unique_id,recency,frequency,monetary_value,mean_freight,review_score_mean,customer_state
0,0000366f3b9a7992bf8c76cfdf3221e2,111,1,141.90,12.00,5.0,SP
1,0000b849f77a49e4a4ce2b2a4ca5be3f,114,1,27.19,8.29,4.0,SP
2,0000f46a3911fa3c0805444483337064,536,1,86.22,17.22,3.0,SC
3,0000f6ccb0745a6a4b88665a16c9f078,320,1,43.62,17.63,4.0,PA
4,0004aac84e0df4da2b147fca70cf8255,287,1,196.89,16.89,5.0,SP


In [5]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# Chargement du fichier qu'on vient de créer à l'étape 1
df_clus_prep = pd.read_csv('../data/olist_clustering.csv')

# 1. Outliers : On plafonne (capping) au 95ème percentile pour stabiliser le K-Means
features_numeric_clus = ['recency', 'frequency', 'monetary_value', 'mean_freight', 'review_score_mean']
for col in ['monetary_value', 'mean_freight', 'frequency']:
    upper_limit = df_clus_prep[col].quantile(0.95)
    df_clus_prep[col] = np.where(df_clus_prep[col] > upper_limit, upper_limit, df_clus_prep[col])

# 2. Encodage : Target Encoding sur la variable textuelle 'customer_state'
state_map = df_clus_prep.groupby('customer_state')['monetary_value'].mean().to_dict()
df_clus_prep['customer_state_encoded'] = df_clus_prep['customer_state'].map(state_map)

# 3. Scaling : Standardisation de toutes nos features numériques
features_to_scale_clus = features_numeric_clus + ['customer_state_encoded']
scaler_clus = StandardScaler()
X_clus_scaled = scaler_clus.fit_transform(df_clus_prep[features_to_scale_clus])

# Création du DataFrame final prêt pour les modèles
df_clustering_final = pd.DataFrame(X_clus_scaled, columns=features_to_scale_clus)
df_clustering_final.insert(0, 'customer_unique_id', df_clus_prep['customer_unique_id'])

# Sauvegarde finale
df_clustering_final.to_csv('../data/olist_ready_clustering.csv', index=False)
print("✅ Dataset Clustering nettoyé, scalé et sauvegardé sous 'olist_ready_clustering.csv' !")

✅ Dataset Clustering nettoyé, scalé et sauvegardé sous 'olist_ready_clustering.csv' !


In [7]:
# Chargement de ton dataset de stocks existant
df_stocks_prep = pd.read_csv('../data/olist_stocks.csv')

# 1. Outliers : On plafonne le volume de ventes hebdomadaire au 99ème percentile
upper_limit_stock = df_stocks_prep['weekly_sales_volume'].quantile(0.99)
df_stocks_prep['weekly_sales_volume'] = np.where(df_stocks_prep['weekly_sales_volume'] > upper_limit_stock, upper_limit_stock, df_stocks_prep['weekly_sales_volume'])

# 2. Scaling : On standardise les variables explicatives (Lags, Moyenne Mobile, Mois)
features_stocks = ['sales_lag_1', 'sales_lag_2', 'sales_moving_avg_4', 'month']
scaler_stocks = StandardScaler()
X_stocks_scaled = scaler_stocks.fit_transform(df_stocks_prep[features_stocks])

# Création du DataFrame final (on garde la cible brute, non scalée !)
df_stocks_final = pd.DataFrame(X_stocks_scaled, columns=features_stocks)
df_stocks_final['weekly_sales_volume'] = df_stocks_prep['weekly_sales_volume']

# Sauvegarde finale
df_stocks_final.to_csv('../data/olist_ready_stocks.csv', index=False)
print("✅ Dataset Stocks nettoyé, scalé et sauvegardé sous 'olist_ready_stocks.csv' !")

✅ Dataset Stocks nettoyé, scalé et sauvegardé sous 'olist_ready_stocks.csv' !
